# Knowledge -  A LightRAG Experiment

This notebook demonstrates how to set up LightRAG, process documents, and perform different types of RAG searches.

In [1]:
#!pip install lightrag-hku pipmaster networkx nano_vectordb asyncio

In [2]:
import asyncio
import os
import inspect
import logging
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete, ollama_embed
from lightrag.utils import EmbeddingFunc

# Set up logging
logging.basicConfig(format="%(levelname)s:%(message)s", level=logging.INFO)

## Initialize LightRAG

Set up LightRAG with Ollama for both the LLM and embeddings. We'll use Deepseek R1:8b as our LLM and nomic-embed-text for embeddings.

In [3]:
WORKING_DIR = "./knowledge"
DOC_DIR = "./documents"

# Create working directory if it doesn't exist
if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

# Initialize LightRAG
rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=ollama_model_complete,
    llm_model_name="deepseek-r1:8b",
    llm_model_max_async=4,
    llm_model_max_token_size=32768,
    llm_model_kwargs={
        "host": "http://localhost:11434",
        "options": {"num_ctx": 32768}
    },
    embedding_func=EmbeddingFunc(
        embedding_dim=768,
        max_token_size=8192,
        func=lambda texts: ollama_embed(
            texts,
            embed_model="nomic-embed-text",
            host="http://localhost:11434"
        ),
    ),
)

INFO:Logger initialized for working directory: ./knowledge
INFO:Load KV json_doc_status_storage with 0 data
INFO:Load KV llm_response_cache with 0 data
INFO:Load KV full_docs with 0 data
INFO:Load KV text_chunks with 0 data
INFO:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './knowledge/vdb_entities.json'} 0 data
INFO:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './knowledge/vdb_relationships.json'} 0 data
INFO:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './knowledge/vdb_chunks.json'} 0 data
INFO:Loaded document status storage with 1 records


## Document Processing Functions

Helper functions to process documents and check for changes.

In [ ]:
import hashlib
from pathlib import Path
from typing import Dict, Set

def get_file_hash(file_path: str) -> str:
    """Calculate SHA-256 hash of file contents."""
    with open(file_path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()

def load_document_hashes(working_dir: str) -> Dict[str, str]:
    """Load existing document hashes from a file."""
    hash_file = os.path.join(working_dir, 'document_hashes.txt')
    if os.path.exists(hash_file):
        hashes = {}
        with open(hash_file, 'r') as f:
            for line in f:
                path, hash_value = line.strip().split('\t')
                hashes[path] = hash_value
        return hashes
    return {}

def save_document_hashes(working_dir: str, hashes: Dict[str, str]):
    """Save document hashes to a file."""
    hash_file = os.path.join(working_dir, 'document_hashes.txt')
    with open(hash_file, 'w') as f:
        for path, hash_value in hashes.items():
            f.write(f"{path}\t{hash_value}\n")

import asyncio
import PyPDF2

async def process_documents(doc_dir: str, rag: LightRAG) -> None:
    """Process all documents in a directory, handling new, modified, and deleted files."""
    # Load existing document hashes
    existing_hashes = load_document_hashes(rag.working_dir)
    current_hashes = {}
    
    # Process all text and pdf files in directory
    for file_path in Path(doc_dir).glob('*.[tp][dx][tf]'):  # matches .txt and .pdf
        file_path_str = str(file_path)
        current_hash = get_file_hash(file_path_str)
        current_hashes[file_path_str] = current_hash
        
        # Check if file is new or modified
        if file_path_str not in existing_hashes or existing_hashes[file_path_str] != current_hash:
            print(f"Processing: {file_path_str}")
            if file_path_str.lower().endswith('.pdf'):
                with open(file_path_str, 'rb') as f:
                    reader = PyPDF2.PdfReader(f)
                    content = '\n'.join(page.extract_text() for page in reader.pages)
            else:  # txt file
                with open(file_path_str, 'r', encoding='utf-8') as f:
                    content = f.read()
            # Insert new content
            await rag.ainsert(content)
    
    # Save current hashes
    save_document_hashes(rag.working_dir, current_hashes)

# Usage
async def main():
    # Process all documents
    await process_documents(DOC_DIR, rag)

# Run the async function
await main()

INFO:All documents have been processed or are duplicates


Processing: documents/Sahih Bukhari (Complete)_djvu.txt
Processing: documents/Pickthall_pickthallquran.pdf


INFO:Processing 1 new unique documents
Generating embeddings: 100%|██████████| 7/7 [08:40<00:00, 74.37s/batch]


## Process Documents

Process all documents in the specified directory.

In [ ]:
# Execute async main function
await main()

# RAG Query Examples

Demonstrate different search modes available in LightRAG.

In [ ]:
query = "What are the main topics covered in these documents?"

# Naive search
print("Naive Search:")
print(rag.query(query, param=QueryParam(mode="naive")))
print("\n" + "-"*50 + "\n")

# Local search
print("Local Search:")
print(rag.query(query, param=QueryParam(mode="local")))
print("\n" + "-"*50 + "\n")

# Global search
print("Global Search:")
print(rag.query(query, param=QueryParam(mode="global")))
print("\n" + "-"*50 + "\n")

# Hybrid search
print("Hybrid Search:")
print(rag.query(query, param=QueryParam(mode="hybrid")))

## Streaming Response Example

Demonstrate how to handle streaming responses from the RAG system.

In [ ]:
async def print_stream(stream):
    async for chunk in stream:
        print(chunk, end="", flush=True)

# Get streaming response
resp = rag.query(
    "What are the main topics covered in these documents?",
    param=QueryParam(mode="hybrid", stream=True)
)

if inspect.isasyncgen(resp):
    await print_stream(resp)
else:
    print(resp)